# Exemplar PSF & Bayer grid

A quick, full-acquisition-free way to see what a single emitter looks like on a Bayer
sensor at a few photon budgets — the same diagnostic the GUI's Simulation tab always
has available (its "Exemplar PSF" section), useful for building intuition about a dye/
filter/photon-budget combination before running a full `simulate_acquisition` (see
`01_simulating_an_acquisition.ipynb`) or a real acquisition.

For each photon level, several independent noise realisations of the same underlying PSF
are rendered side by side — the same true signal, sampled repeatedly, so you can see how
much frame-to-frame variation to expect at that brightness.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import pyS3M.SpectralFunctions as SpectralFunctions
from pyS3M.PSFFunctions import PSF_Functions
import pyS3M.PlottingBase as PlottingBase

# Parameters -- match the GUI Simulation panel's own defaults
DYE = "Cy3B"
FILTERS: list[str] = []       # no emission filter
BG_PHOTONS = 10.0
NA = 1.49
PIXEL_SIZE_NM = 69.0
READ_NOISE_E = 1.0
PEAK_QE = 0.7
N_REP = 5
PHOTON_LEVELS = [200, 500, 1_000, 5_000]

plotter = PlottingBase.PublicationPlotter()

## Spectral split — how the dye's emission divides across the Bayer R/G/B channels

`get_pixel_fractions_dye_and_filters` combines the dye's emission spectrum with the
camera's per-channel quantum efficiency curves (and any emission filters) to give the
fraction of detected photons landing in each Bayer colour — this, not position, is what
`unmix_channels` (`../analyses/05_channel_unmixing.ipynb`) separates dye populations by.

In [2]:
S_F = SpectralFunctions.Spectral_Funcs()
R_qe, G_qe, B_qe, wl = S_F.getpixelefficiency()

raw_peak = max(R_qe.max(), G_qe.max(), B_qe.max())
scale = PEAK_QE / raw_peak if raw_peak > 0 else 1.0
R_sc, G_sc, B_sc = R_qe * scale, G_qe * scale, B_qe * scale
pixel_QYs = np.vstack([B_sc, G_sc, R_sc])  # (n_channels, n_wavelengths), BGR order

avg_wl, fracs = S_F.get_pixel_fractions_dye_and_filters(
    dyes=[DYE], filters=FILTERS if FILTERS else None,
    wavelength=wl, pixel_QYs=pixel_QYs, normalized=True,
)
b_frac, g_frac, r_frac = fracs
b_eff, g_eff, r_eff = b_frac * PEAK_QE, g_frac * PEAK_QE, r_frac * PEAK_QE
print(f"{DYE}: avg emission {avg_wl:.0f} nm, colour split R={r_frac:.2f} G={g_frac:.2f} B={b_frac:.2f}")

Cy3B: avg emission 601 nm, colour split R=0.47 G=0.49 B=0.03


## PSF kernel and Bayer masks

`PSF_Functions.sigma_PSF` gives the diffraction-limited PSF width from the emission
wavelength and NA — the same function `simulate_acquisition` uses internally for the PSF
sigma reported as "recommended Peak λ" in every `test_tiffs/` fixture's README.

In [3]:
sigma_nm = PSF_Functions.sigma_PSF(float(avg_wl) * 1e-9, NA) * 1e9  # m -> nm
sigma_px = sigma_nm / PIXEL_SIZE_NM
print(f"PSF sigma: {sigma_nm:.0f} nm ({sigma_px:.2f} camera px)")

patch_size = 13
c = patch_size // 2
yy, xx = np.mgrid[:patch_size, :patch_size]
psf = np.exp(-((xx - c) ** 2 + (yy - c) ** 2) / (2.0 * sigma_px ** 2))
psf /= psf.sum()

# RGGB Bayer masks over the patch
r_mask = (yy % 2 == 0) & (xx % 2 == 0)
g_mask = ((yy % 2 == 0) & (xx % 2 == 1)) | ((yy % 2 == 1) & (xx % 2 == 0))
b_mask = (yy % 2 == 1) & (xx % 2 == 1)

PSF sigma: 91 nm (1.32 camera px)


## Render the grid — one row per photon level, `N_REP` independent noise draws each

Each pixel's signal is Poisson-sampled from the PSF-weighted, colour-split photon count
plus background, with additive Gaussian read noise — the same forward model
`simulate_acquisition`/`gen_camera_image_stack` use for a full acquisition, just for a
single isolated emitter patch instead of a whole frame stack.

In [ ]:
rng = np.random.default_rng(42)
n_rows = len(PHOTON_LEVELS)
fig, axes = plotter.two_column_plot(nrows=n_rows, ncols=N_REP)

for row_idx, n_ph in enumerate(PHOTON_LEVELS):
    row_patches = []
    for col_idx in range(N_REP):
        bayer = np.zeros((patch_size, patch_size))
        for mask, eff in ((r_mask, r_eff), (g_mask, g_eff), (b_mask, b_eff)):
            n_px = int(mask.sum())
            sig = rng.poisson(n_ph * eff * psf[mask] + BG_PHOTONS).astype(float)
            sig += rng.normal(0.0, READ_NOISE_E, size=n_px)
            bayer[mask] = sig - BG_PHOTONS
        row_patches.append(bayer)

    all_vals = np.concatenate([p.ravel() for p in row_patches])
    vmin, vmax = np.percentile(all_vals, [0.1, 99.9])
    if vmax <= vmin:
        vmax = vmin + 1.0

    for col_idx, bayer in enumerate(row_patches):
        ax = axes[row_idx, col_idx]
        # image_plot's show_axes=False hides the ylabel too (ax.axis("off")), which we
        # need here for the per-row photon-count label -- ticks-off-but-label-on isn't
        # something it supports, so this tile stays direct matplotlib.
        ax.imshow(bayer, cmap="gray", vmin=vmin, vmax=vmax, interpolation="nearest")
        ax.set_xticks([])
        ax.set_yticks([])
        if col_idx == 0:
            ax.set_ylabel(f"{n_ph:,} ph", rotation=0, labelpad=32, va="center")

fig.suptitle(
    f"{DYE}  |  λ={avg_wl:.0f} nm  σ={sigma_nm:.0f} nm  "
    f"QE={PEAK_QE:.2f}  BG={BG_PHOTONS:.0f} ph/px  RN={READ_NOISE_E:.1f} e⁻"
)
plt.tight_layout()
plt.show()

## Next steps

This and `01_simulating_an_acquisition.ipynb` cover the full simulation backend used to
generate every `test_tiffs/` fixture across `../analyses/` — see
`claude/generate_test_fixtures.py` in the repository for how each specific fixture was
built, if you want to create a new one of your own.